# GRPO: Group Relative Policy Optimization for Language Models

This notebook implements **GRPO** (Shao et al., 2024), the RL algorithm behind
**DeepSeek-R1**. GRPO simplifies PPO for language model alignment by eliminating the
critic network entirely — using group-relative reward normalization instead.

We will:
1. Explain why PPO is expensive for LLMs (the critic problem)
2. Derive GRPO's key insight: group-relative advantages
3. Understand the KL penalty for preventing reward hacking
4. Implement GRPO from scratch on GPT-2
5. Show TRL's production `GRPOTrainer`
6. Compare GRPO, PPO, and DPO

**References:**
- Shao et al. (2024). *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models.* https://arxiv.org/abs/2402.03300
- DeepSeek-AI (2025). *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via RL.* https://arxiv.org/abs/2501.12948

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.device import set_seed

set_seed(42)
device = "cpu"  # From-scratch section runs on CPU; TRL section needs CUDA

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"Transformers available")
except ImportError:
    print("transformers not installed")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

Set seed for reproducibility: 42


/Users/maximilianruess/Documents/GitHub/DeepLearning_101/dl_101/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers available
PyTorch version: 2.11.0
Device: cpu


## 1. PPO's Problem for LLMs

PPO (from our previous notebook) uses an **actor-critic** architecture:
- **Actor** (policy): the language model that generates text
- **Critic** (value network): estimates expected future reward for each state

For LLMs, the critic is typically **the same size as the policy**. For a 7B model:

| Component | Memory |
|---|---|
| Policy model | ~14 GB (fp16) |
| Reference model (frozen) | ~14 GB (fp16) |
| Critic/value model | ~14 GB (fp16) |
| **Total** | **~42 GB** |

That's 3 full copies of the model. The critic also needs to be trained well — a bad critic
gives bad advantage estimates, which makes the policy update noisy.

**Can we eliminate the critic entirely?**

## 2. GRPO's Key Insight

Instead of learning a value baseline (critic), GRPO uses a **statistical baseline**:
the mean reward of multiple completions for the same prompt.

For each prompt $p$, generate $G$ completions $\{o_1, o_2, ..., o_G\}$ and score them
with a reward function $R(p, o_i)$. The group-relative advantage is:

$\hat{A}_i = \frac{R(p, o_i) - \mu_G}{\sigma_G}$

where $\mu_G = \frac{1}{G}\sum_{j=1}^G R(p, o_j)$ and $\sigma_G = \text{std}(R(p, o_1), ..., R(p, o_G))$.

**Intuition:** We don't need to know the absolute value of a state. We just need to know
which completions are *relatively* better within the group.

## 3. GRPO vs PPO

| | PPO | GRPO |
|---|---|---|
| Critic/Value network | Yes (same size as policy) | **No** |
| Advantage estimation | GAE (temporal difference) | Group-relative normalization |
| Memory (7B model) | ~42 GB (3 models) | **~28 GB (2 models)** |
| Completions per prompt | 1 | G (typically 4-16) |
| Baseline | Learned value function $V(s)$ | Sample mean $\mu_G$ |
| KL penalty | Optional | Required (prevents reward hacking) |

## 4. The GRPO Objective

GRPO uses the same clipped surrogate as PPO, but with group-relative advantages
and an explicit KL penalty:

$L(\theta) = \mathbb{E}_{p, \{o_i\}} \left[ \frac{1}{G} \sum_{i=1}^G \left( \min\left(r_i \hat{A}_i, \text{clip}(r_i, 1-\epsilon, 1+\epsilon) \hat{A}_i\right) - \beta \cdot D_{KL}(\pi_\theta \| \pi_{ref}) \right) \right]$

where $r_i = \frac{\pi_\theta(o_i | p)}{\pi_{\theta_{old}}(o_i | p)}$ is the probability ratio.

**KL penalty** $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$ prevents the policy from drifting too
far from the reference model. Without it, the policy can exploit the reward function
in degenerate ways ("reward hacking").

## 5. Group-Relative Advantages

In [2]:
def compute_group_advantages(rewards, eps=1e-8):
    """
    GRPO advantage: normalize rewards within the group.
    A_i = (r_i - mean(r)) / (std(r) + eps)
    """
    rewards = np.array(rewards, dtype=np.float32)
    mean = rewards.mean()
    std = rewards.std()
    return (rewards - mean) / (std + eps)


# Demo: 8 completions of the same prompt with different rewards
rewards = [0.2, 0.8, 0.5, 0.1, 0.9, 0.3, 0.7, 0.6]
advantages = compute_group_advantages(rewards)

print("Reward → Advantage (group-relative):")
for r, a in sorted(zip(rewards, advantages), key=lambda x: x[0]):
    bar = "+" * int(max(0, a) * 10) if a > 0 else "-" * int(abs(a) * 10)
    print(f"  reward={r:.1f} → advantage={a:+.2f}  {bar}")

print(f"\nMean advantage: {advantages.mean():.6f} (should be ~0)")
print(f"Std advantage:  {advantages.std():.6f} (should be ~1)")

Reward → Advantage (group-relative):
  reward=0.1 → advantage=-1.52  ---------------
  reward=0.2 → advantage=-1.15  -----------
  reward=0.3 → advantage=-0.78  -------
  reward=0.5 → advantage=-0.05  
  reward=0.6 → advantage=+0.32  +++
  reward=0.7 → advantage=+0.69  ++++++
  reward=0.8 → advantage=+1.06  ++++++++++
  reward=0.9 → advantage=+1.43  ++++++++++++++

Mean advantage: 0.000000 (should be ~0)
Std advantage:  1.000000 (should be ~1)


## 6. Reward Functions for LLMs

GRPO is reward-function agnostic. The reward function scores each completion and GRPO
optimizes the policy to produce higher-reward completions. Common reward functions:

| Reward Type | Example | Used In |
|---|---|---|
| Accuracy | Math problem correctness | DeepSeek-Math |
| Format compliance | Follows XML/JSON format | DeepSeek-R1 |
| Length penalty | Discourages verbose output | Most RLHF systems |
| Human preference | Reward model trained on preferences | ChatGPT (PPO-based) |

For our demo, we'll use **math accuracy** on [GSM8K](https://huggingface.co/datasets/openai/gsm8k)
— grade school math problems with verifiable answers. This is exactly how DeepSeek-Math
used GRPO: the reward is 1.0 if the model's final answer matches the ground truth, 0.0 otherwise.

## 7. Reference Model & KL Penalty

The **reference model** $\pi_{ref}$ is a frozen copy of the initial policy.
The KL divergence measures how far the current policy has drifted:

$D_{KL}(\pi_\theta \| \pi_{ref}) = \sum_t \left[ \log \pi_\theta(a_t | s_t) - \log \pi_{ref}(a_t | s_t) \right]$

**Why this matters:** Without the KL penalty, the policy can find degenerate strategies
that exploit the reward function. For example, if we reward length, the model might
repeat the same word forever. The KL term keeps the model close to its pretrained behavior.

In [3]:
# Demo: per-token KL divergence
def compute_per_token_kl(logprobs_policy, logprobs_ref):
    """Approximate KL(policy || ref) per token."""
    return logprobs_policy - logprobs_ref


# Simulate log-probs for 10 tokens
logp_policy = torch.tensor([-1.2, -0.8, -2.1, -0.5, -1.5, -0.9, -1.8, -0.3, -1.1, -0.7])
logp_ref    = torch.tensor([-1.0, -1.0, -2.0, -0.6, -1.4, -1.0, -2.0, -0.5, -1.0, -0.8])

kl = compute_per_token_kl(logp_policy, logp_ref)
print(f"Per-token KL: {kl.numpy()}")
print(f"Total KL:     {kl.sum().item():.4f}")
print(f"Mean KL:      {kl.mean().item():.4f}")
print("\nPositive KL = policy assigns higher prob than reference (drifting)")
print("Negative KL = policy assigns lower prob (also drifting, other direction)")

Per-token KL: [-0.20000005  0.19999999 -0.0999999   0.10000002 -0.10000002  0.10000002
  0.20000005  0.19999999 -0.10000002  0.10000002]
Total KL:     0.4000
Mean KL:      0.0400

Positive KL = policy assigns higher prob than reference (drifting)
Negative KL = policy assigns lower prob (also drifting, other direction)


## 8. GRPO Algorithm

```
Initialize policy π_θ, reference model π_ref = copy(π_θ)
For each iteration:
    For each prompt p in batch:
        1. Generate G completions: {o_1, ..., o_G} ~ π_θ(·|p)
        2. Score: r_i = R(p, o_i) for each completion
        3. Compute group advantages: A_i = (r_i - mean(r)) / std(r)
        4. Record old log-probs: log π_θ_old(o_i|p)
    
    For K epochs over the collected data:
        For each mini-batch:
            ratio = π_θ(o|p) / π_θ_old(o|p)
            L_clip = min(ratio * A, clip(ratio, 1-ε, 1+ε) * A)
            L_kl = β * (log π_θ(o|p) - log π_ref(o|p))
            loss = -L_clip + L_kl
            Gradient step on θ
```

In [4]:
class SimpleGRPOTrainer:
    """
    Educational GRPO implementation on GPT-2.
    NOT production-grade — use TRL's GRPOTrainer for real work.
    """

    def __init__(self, model_name="gpt2", lr=1e-5, clip_epsilon=0.2,
                 kl_coeff=0.1, group_size=4, max_new_tokens=32):
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.ref_model = copy.deepcopy(self.model)
        self.ref_model.eval()
        for p in self.ref_model.parameters():
            p.requires_grad = False

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.clip_epsilon = clip_epsilon
        self.kl_coeff = kl_coeff
        self.group_size = group_size
        self.max_new_tokens = max_new_tokens

        self.losses = []
        self.rewards_history = []

    def generate_completions(self, prompt, num_completions):
        """Generate G completions for a prompt."""
        inputs = self.tokenizer(prompt, return_tensors="pt")
        input_ids = inputs["input_ids"]
        prompt_len = input_ids.shape[1]

        completions = []
        self.model.eval()
        with torch.no_grad():
            for _ in range(num_completions):
                output = self.model.generate(
                    input_ids,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=True,
                    temperature=0.8,
                    top_k=50,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
                completion_ids = output[0][prompt_len:]  # only the generated part
                completion_text = self.tokenizer.decode(completion_ids, skip_special_tokens=True)
                completions.append({
                    "text": completion_text,
                    "input_ids": output[0],
                    "prompt_len": prompt_len,
                })

        return completions

    def get_log_probs(self, model, input_ids, prompt_len):
        """Get per-token log probs for the completion part."""
        outputs = model(input_ids.unsqueeze(0))
        logits = outputs.logits[0, prompt_len-1:-1]  # shifted by 1 for next-token prediction
        target_ids = input_ids[prompt_len:]
        log_probs = F.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)
        return token_log_probs

    def train_step(self, prompt, reward_fn):
        """One GRPO training step on a single prompt."""
        # 1. Generate G completions
        completions = self.generate_completions(prompt, self.group_size)

        # 2. Score completions
        rewards = [reward_fn(prompt, c["text"]) for c in completions]
        self.rewards_history.append(np.mean(rewards))

        # 3. Compute group-relative advantages
        advantages = compute_group_advantages(rewards)

        # 4. Get old log-probs (before update)
        self.model.eval()
        old_log_probs_list = []
        with torch.no_grad():
            for c in completions:
                old_lp = self.get_log_probs(self.model, c["input_ids"], c["prompt_len"])
                old_log_probs_list.append(old_lp)

        # 5. Policy update
        self.model.train()
        total_loss = 0.0

        for c, adv, old_lp in zip(completions, advantages, old_log_probs_list):
            # Current log-probs
            new_lp = self.get_log_probs(self.model, c["input_ids"], c["prompt_len"])

            # Reference log-probs (for KL)
            with torch.no_grad():
                ref_lp = self.get_log_probs(self.ref_model, c["input_ids"], c["prompt_len"])

            # Probability ratio
            ratio = torch.exp(new_lp - old_lp)

            # Clipped surrogate
            adv_tensor = torch.tensor(adv, dtype=torch.float32)
            surr1 = ratio * adv_tensor
            surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * adv_tensor
            clip_loss = -torch.min(surr1, surr2).mean()

            # KL penalty
            kl = (new_lp - ref_lp).mean()
            kl_loss = self.kl_coeff * kl

            loss = clip_loss + kl_loss
            total_loss += loss

        # Average over group and step
        total_loss = total_loss / self.group_size
        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        self.losses.append(total_loss.item())
        return np.mean(rewards)


print("SimpleGRPOTrainer ready.")

SimpleGRPOTrainer ready.


In [9]:
import re

def extract_last_number(text):
    """Extract the last number from a string (handles commas, negatives, decimals)."""
    numbers = re.findall(r'-?\d[\d,]*\.?\d*', text)
    if not numbers:
        return None
    return float(numbers[-1].replace(',', ''))


def math_reward(prompt, completion, answer):
    """
    Reward function for math problems.
    Returns 1.0 if the completion contains the correct answer, 0.0 otherwise.
    """
    predicted = extract_last_number(completion)
    if predicted is None:
        return 0.0
    return 1.0 if abs(predicted - answer) < 1e-3 else 0.0


# Test the reward function
test_cases = [
    ("The answer is 42.", 42),
    ("I think it's about 15 apples.", 15),
    ("Let me think... 3 + 4 = 7. So the answer is 7.", 7),
    ("I'm not sure about this problem.", 7),
    ("The total cost is $12.50", 12.5),
]
print("Testing math_reward:")
for text, answer in test_cases:
    score = math_reward("", text, answer)
    print(f"  reward={score:.0f} | answer={answer} | '{text[:50]}'")

Testing math_reward:
  reward=1 | answer=42 | 'The answer is 42.'
  reward=1 | answer=15 | 'I think it's about 15 apples.'
  reward=1 | answer=7 | 'Let me think... 3 + 4 = 7. So the answer is 7.'
  reward=0 | answer=7 | 'I'm not sure about this problem.'
  reward=1 | answer=12.5 | 'The total cost is $12.50'


In [10]:
# Train GRPO on GPT-2 with math problems (from scratch, runs on CPU)
# This is slow on CPU — just a few steps for demonstration
print("Training GRPO on GPT-2 with GSM8K math problems (educational demo, CPU)...")
print("This is intentionally small-scale. Use TRL + GPU for real training.\n")

# Sample GSM8K-style problems with known answers
math_problems = [
    {"prompt": "Q: Janet has 3 apples and buys 5 more. How many apples does she have?\nA: Let me solve this step by step.", "answer": 8},
    {"prompt": "Q: A store has 20 shirts. If 7 are sold, how many remain?\nA: Let me solve this step by step.", "answer": 13},
    {"prompt": "Q: Tom has 4 boxes with 6 pencils each. How many pencils total?\nA: Let me solve this step by step.", "answer": 24},
    {"prompt": "Q: A class has 15 boys and 12 girls. How many students total?\nA: Let me solve this step by step.", "answer": 27},
    {"prompt": "Q: Sarah had 50 dollars and spent 23 dollars. How much is left?\nA: Let me solve this step by step.", "answer": 27},
]

set_seed(42)
trainer = SimpleGRPOTrainer(
    model_name="gpt2",
    lr=1e-5,
    clip_epsilon=0.2,
    kl_coeff=0.1,
    group_size=4,
    max_new_tokens=32,
)

num_steps = 6
for step in range(num_steps):
    problem = math_problems[step % len(math_problems)]
    reward_fn = lambda prompt, completion, ans=problem["answer"]: math_reward(prompt, completion, ans)
    avg_reward = trainer.train_step(problem["prompt"], reward_fn)
    print(f"Step {step+1}/{num_steps} | Answer: {problem['answer']} | "
          f"Avg reward: {avg_reward:.3f} | Loss: {trainer.losses[-1]:.4f}")

print("\nDone. (In production, train for thousands of steps on GPU with full GSM8K.)")

Training GRPO on GPT-2 with GSM8K math problems (educational demo, CPU)...
This is intentionally small-scale. Use TRL + GPU for real training.

Set seed for reproducibility: 42


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 14581.69it/s]


Step 1/6 | Answer: 8 | Avg reward: 0.000 | Loss: -0.0324
Step 2/6 | Answer: 13 | Avg reward: 0.000 | Loss: -0.0315
Step 3/6 | Answer: 24 | Avg reward: 0.000 | Loss: -0.0353
Step 4/6 | Answer: 27 | Avg reward: 0.000 | Loss: -0.0256
Step 5/6 | Answer: 27 | Avg reward: 0.000 | Loss: -0.0398
Step 6/6 | Answer: 8 | Avg reward: 0.000 | Loss: -0.0313

Done. (In production, train for thousands of steps on GPU with full GSM8K.)


## 9. TRL: The Production Way

In practice, use [TRL](https://huggingface.co/docs/trl)'s `GRPOTrainer` which handles:
- Efficient batched generation
- KL penalty computation
- Gradient accumulation and mixed precision
- Distributed training
- Integration with PEFT/LoRA

```bash
pip install -e ".[llm]"  # installs trl, peft, accelerate, etc.
```

**Requires CUDA GPU.** If running locally on Mac, skip to Section 11 (Modal).

In [11]:
# TRL GRPOTrainer with GSM8K (CUDA only)
if torch.cuda.is_available():
    from trl import GRPOTrainer, GRPOConfig
    from datasets import load_dataset

    # Model
    model = AutoModelForCausalLM.from_pretrained("gpt2")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # GSM8K dataset
    gsm8k = load_dataset("openai/gsm8k", "main", split="train[:200]")

    # Format prompts for the model
    def format_gsm8k(example):
        example["prompt"] = f"Q: {example['question']}\nA: Let me solve this step by step."
        # Extract the numeric answer from GSM8K's "#### <number>" format
        answer_str = example["answer"].split("####")[-1].strip()
        example["ground_truth"] = float(answer_str.replace(",", ""))
        return example

    gsm8k = gsm8k.map(format_gsm8k)

    # Math accuracy reward for TRL
    def math_reward_fn(completions, ground_truth, **kwargs):
        """TRL reward: 1.0 if final number matches ground truth, 0.0 otherwise."""
        rewards = []
        for completion, answer in zip(completions, ground_truth):
            predicted = extract_last_number(completion)
            if predicted is not None and abs(predicted - answer) < 1e-3:
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        return rewards

    config = GRPOConfig(
        output_dir="./grpo-gpt2-math",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        num_generations=4,
        max_completion_length=128,
        learning_rate=1e-5,
        logging_steps=10,
        report_to="none",
    )

    trl_trainer = GRPOTrainer(
        model=model,
        args=config,
        train_dataset=gsm8k,
        processing_class=tokenizer,
        reward_funcs=math_reward_fn,
    )

    trl_trainer.train()
    print("TRL GRPOTrainer training on GSM8K complete!")
else:
    print(f"Device: {device}")
    print("TRL GRPOTrainer requires CUDA. Skip to Section 10 for Modal GPU.")
    print()
    print("The setup uses GSM8K (grade school math):")
    print("  dataset = load_dataset('openai/gsm8k', 'main', split='train[:200]')")
    print("  reward: 1.0 if model's final number matches ground truth, 0.0 otherwise")
    print("  GRPOConfig(num_generations=4, max_completion_length=128, lr=1e-5)")

Device: cpu
TRL GRPOTrainer requires CUDA. Skip to Section 10 for Modal GPU.

The setup uses GSM8K (grade school math):
  dataset = load_dataset('openai/gsm8k', 'main', split='train[:200]')
  reward: 1.0 if model's final number matches ground truth, 0.0 otherwise
  GRPOConfig(num_generations=4, max_completion_length=128, lr=1e-5)


In [12]:
# Generate and compare: test the from-scratch trainer on a math problem
print("Sample completions after GRPO training:\n")
trainer.model.eval()
test_problems = [
    "Q: Janet has 3 apples and buys 5 more. How many apples does she have?\nA: Let me solve this step by step.",
    "Q: A store has 20 shirts. If 7 are sold, how many remain?\nA: Let me solve this step by step.",
]

for prompt in test_problems:
    inputs = trainer.tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = trainer.model.generate(
            inputs["input_ids"],
            max_new_tokens=40,
            do_sample=True,
            temperature=0.7,
            pad_token_id=trainer.tokenizer.eos_token_id,
        )
    text = trainer.tokenizer.decode(output[0], skip_special_tokens=True)
    question = prompt.split("\n")[0]
    completion = text[len(prompt):]
    print(f"{question}")
    print(f"Completion: {completion.strip()}")
    print()

Sample completions after GRPO training:

Q: Janet has 3 apples and buys 5 more. How many apples does she have?
Completion: The equation is 1.
Q: How many apples does she have and buys 5 more. How many apples does she have?
A: The 2nd equation is 9.
Q: How

Q: A store has 20 shirts. If 7 are sold, how many remain?
Completion: Q: How much are you selling?
A: Let me sell for 99 cents.
Q: What do you expect to buy in the store?
A: I don't know.



## 10. Run on GPU (Modal)

For real GRPO training, use [Modal](https://modal.com) to run on a remote GPU.

```bash
pip install modal
modal token set
```

In [ ]:
MODAL_SCRIPT = '''
import modal

image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("torch", "transformers", "trl", "peft", "accelerate", "datasets", "tqdm")
)

app = modal.App("dl101-grpo")

@app.function(gpu="A100", image=image, timeout=3600)
def train_grpo_gsm8k() -> dict:
    import re
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import GRPOTrainer, GRPOConfig
    from datasets import load_dataset

    print(f"GPU: {torch.cuda.get_device_name()}")

    model = AutoModelForCausalLM.from_pretrained("gpt2")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # Load and format GSM8K
    gsm8k = load_dataset("openai/gsm8k", "main", split="train[:500]")

    def format_gsm8k(example):
        example["prompt"] = f"Q: {example[\'question\']}\\nA: Let me solve this step by step."
        answer_str = example["answer"].split("####")[-1].strip()
        example["ground_truth"] = float(answer_str.replace(",", ""))
        return example

    gsm8k = gsm8k.map(format_gsm8k)

    def extract_last_number(text):
        numbers = re.findall(r\'-?[\\d,]+\\.?\\d*\', text)
        return float(numbers[-1].replace(\',\', \'\')) if numbers else None

    def math_reward_fn(completions, ground_truth, **kwargs):
        rewards = []
        for completion, answer in zip(completions, ground_truth):
            predicted = extract_last_number(completion)
            if predicted is not None and abs(predicted - answer) < 1e-3:
                rewards.append(1.0)
            else:
                rewards.append(0.0)
        return rewards

    config = GRPOConfig(
        output_dir="/tmp/grpo-output",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        num_generations=4,
        max_completion_length=128,
        learning_rate=1e-5,
        logging_steps=10,
        report_to="none",
    )

    trainer = GRPOTrainer(
        model=model, args=config, train_dataset=gsm8k,
        processing_class=tokenizer, reward_funcs=math_reward_fn,
    )

    trainer.train()

    return {
        "gpu_name": torch.cuda.get_device_name(),
        "peak_memory_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "train_loss": trainer.state.log_history[-1].get("loss", None),
    }
'''

print("Modal script defined.")
print("Save as .py and run: modal run <file>.py")

## 11. Reusable Implementation

The core GRPO utility functions are in `src/models/grpo.py`:

```python
from src.models.grpo import compute_group_advantages, compute_per_token_kl
```

In [ ]:
from src.models.grpo import compute_group_advantages as cga, compute_per_token_kl as kl

# Verify
adv = cga([1.0, 2.0, 3.0, 4.0])
print(f"Group advantages: {adv}")
print(f"Mean ~0: {adv.mean():.6f}, Std ~1: {adv.std():.4f}")
print("All src/ imports verified.")

## 12. Key Takeaways

1. **GRPO eliminates the critic network.** Instead of learning a value function, it uses the group mean reward as a baseline. This saves ~33% of memory for LLM training.

2. **Group-relative advantages are simple and effective.** For G completions per prompt: $\hat{A}_i = (r_i - \mu_G) / \sigma_G$. Better completions get positive advantage, worse ones get negative.

3. **The KL penalty prevents reward hacking.** Without it, the policy can exploit the reward function in degenerate ways. The penalty keeps the policy close to the pretrained reference model.

4. **GRPO powers DeepSeek-R1.** DeepSeek used GRPO to train reasoning capabilities into their LLM, producing chain-of-thought reasoning without explicit supervision.

5. **Use TRL's GRPOTrainer for production.** The from-scratch implementation is educational; TRL handles batched generation, distributed training, and PEFT integration.

6. **Comparison of LLM alignment methods:**

| | PPO | GRPO | DPO |
|---|---|---|---|
| Requires critic | Yes | **No** | No |
| Requires reward model | Yes | Yes | **No** (uses preferences) |
| Online generation | Yes | Yes | **No** (offline) |
| Memory (7B model) | ~42 GB | ~28 GB | ~28 GB |
| Used by | ChatGPT | DeepSeek-R1 | Llama, Zephyr |

### Further Reading

- Shao et al. (2024). *DeepSeekMath.* https://arxiv.org/abs/2402.03300
- DeepSeek-AI (2025). *DeepSeek-R1.* https://arxiv.org/abs/2501.12948
- TRL GRPO docs: https://huggingface.co/docs/trl/grpo_trainer
- Rafailov et al. (2023). *DPO: Direct Preference Optimization.* https://arxiv.org/abs/2305.18290